# Notebook 2: The Event Loop

**What you'll learn:**
- What the event loop is and why it exists
- How to observe the event loop with logging
- What happens when there are no tools (simple path)
- What happens when there ARE tools (tool_use -> recurse -> end_turn)
- What multiple tool calls look like
- Stop reasons: end_turn, tool_use, max_tokens

**Prerequisite:** Complete [NB1_Your_First_Agent.ipynb](./NB1_Your_First_Agent.ipynb)

**Companion reading:** `02-the-event-loop.md`

---
## What is an Event Loop?

An **event loop** is a program that waits for things to happen and responds to them. Think of a restaurant waiter:

1. Take order (event)
2. Go to kitchen (process)
3. Bring food back (result)
4. Take dessert order (next event)
5. Repeat...

The SDK's event loop is similar but **not persistent** -- it runs once per agent call, then stops. Here's the state machine:

```
                    +------------------+
                    |   START          |
                    |   Call Model     |
                    +------------------+
                            |
                            v
                    +------------------+
                    |  Model returns   |
                    |  with stop_reason|
                    +------------------+
                       /    |    \
                      /     |     \
              "end_turn" "tool_use" "max_tokens"
                    /       |          \
                   v        v           v
            +---------+ +----------+ +----------+
            |  DONE   | | Execute  | |  ERROR   |
            | Return  | |  Tools   | |  Throw   |
            | result  | +----------+ +----------+
            +---------+      |
                             v
                      +----------+
                      | RECURSE  |
                      | Go back  |
                      | to START |
                      +----------+
```

**Key insight:** When the model wants to use a tool, the event loop executes the tool and then **calls itself again** (recurses). This lets the model see the tool result and decide what to do next.

**Source:** `src/strands/event_loop/event_loop.py:78`

In [ ]:
# ============================================================
# SETUP: Enable logging so we can see the event loop in action
# ============================================================

import logging
import json

# Python's logging module lets us see internal messages from the SDK.
# The SDK logs key events: model calls, tool execution, stop reasons.
#
# logging.DEBUG    = most detailed (everything)
# logging.INFO     = normal events
# logging.WARNING  = potential problems
# logging.ERROR    = things that went wrong

# Set up logging for the strands.event_loop module.
# This will print messages from event_loop.py to the console.
logger = logging.getLogger("strands.event_loop")
logger.setLevel(logging.DEBUG)

# Also add logging for the agent module
agent_logger = logging.getLogger("strands.agent")
agent_logger.setLevel(logging.DEBUG)

# Create a handler that prints to the console with timestamps
handler = logging.StreamHandler()
handler.setLevel(logging.DEBUG)
formatter = logging.Formatter('%(name)s | %(message)s')
handler.setFormatter(formatter)

# Add handler to both loggers (avoid duplicates)
logger.handlers = [handler]
agent_logger.handlers = [handler]

print("Logging enabled for strands.event_loop and strands.agent")

---
## Experiment 1: Agent Without Tools (Simple Path)

When the agent has no tools, the event loop is simple:
1. Call the model
2. Model responds with `stop_reason = "end_turn"`
3. Done!

No recursion, no tool execution. Just one cycle.

In [ ]:
# ============================================================
# EXPERIMENT 1: No tools -- single event loop cycle
# ============================================================

from strands import Agent

# Create an agent with no tools.
# callback_handler=None disables real-time streaming output,
# so we only see the logging messages and our own prints.
agent_no_tools = Agent(callback_handler=None)

# Call the agent. Watch the logging output.
# You should see:
#   - Model being called
#   - stop_reason = "end_turn"
#   - No tool execution
print("--- Calling agent (no tools) ---")
result = agent_no_tools("What is the capital of France? Answer in one word.")
print(f"\n--- Result ---")
print(f"stop_reason: {result.stop_reason}")
print(f"response: {result}")

In [ ]:
# ============================================================
# Inspect messages after Experiment 1
# ============================================================

# With no tools, there should be exactly 2 messages:
#   [0] user:      "What is the capital of France?"
#   [1] assistant:  "Paris"

print(f"Message count: {len(agent_no_tools.messages)}")
print()

for i, msg in enumerate(agent_no_tools.messages):
    text = ""
    for block in msg['content']:
        if 'text' in block:
            text = block['text'][:100]
            break
    print(f"[{i}] {msg['role']:10s} | {text}")

---
## Experiment 2: Agent With a Tool (The Recursion!)

Now let's add a tool. When the model decides to use a tool, the event loop does more work:

**Cycle 1:**
1. Call the model
2. Model responds with `stop_reason = "tool_use"` ("I want to use the calculator")
3. Event loop executes the tool
4. Tool result is appended to messages
5. **RECURSE** -- call event_loop_cycle again

**Cycle 2:**
1. Call the model again (it can now see the tool result)
2. Model responds with `stop_reason = "end_turn"` ("Here's the answer")
3. Done!

This creates **4 messages** instead of 2.

In [ ]:
# ============================================================
# Define a simple calculator tool
# ============================================================

from strands import tool

# The @tool decorator transforms this function into an AgentTool.
# It reads the function name, docstring, and type hints to create
# a JSON schema (tool_spec) that the model can understand.
#
# The model will see:
#   - Name: "calculator"
#   - Description: "Perform basic math..."
#   - Parameters: expression (string, required)

@tool
def calculator(expression: str) -> str:
    """Perform basic math calculations.
    
    Args:
        expression: A math expression to evaluate, e.g. "17 * 23"
    """
    # eval() runs a string as Python code.
    # In production, you'd want a safer math parser!
    result = eval(expression)
    return str(result)

print(f"Tool name: {calculator.tool_name}")
print(f"Tool type: {calculator.tool_type}")

In [ ]:
# ============================================================
# EXPERIMENT 2: With a tool -- two event loop cycles
# ============================================================

# Create an agent with the calculator tool.
agent_with_tool = Agent(
    tools=[calculator],          # Register the calculator tool
    callback_handler=None,       # Disable streaming output
)

# Ask a math question. The model should:
#   1. Decide to call the calculator tool
#   2. Wait for the result
#   3. Formulate the final answer
#
# Watch the logging output for:
#   - Cycle 1: model call -> stop_reason="tool_use" -> execute calculator
#   - Cycle 2: model call -> stop_reason="end_turn" -> done

print("--- Calling agent (with tool) ---")
result = agent_with_tool("What is 17 * 23?")
print(f"\n--- Result ---")
print(f"stop_reason: {result.stop_reason}")
print(f"response: {result}")

In [ ]:
# ============================================================
# Inspect messages after Experiment 2
# ============================================================

# With one tool call, there should be 4 messages:
#   [0] user:      "What is 17 * 23?"
#   [1] assistant:  "I'll calculate..." + toolUse request
#   [2] user:      toolResult ("391")
#   [3] assistant:  "17 * 23 = 391"
#
# Notice: tool results are sent as "user" role messages.
# This is because the model only reads user messages and its own messages.
# Tool results must come from the "user" side.

print(f"Message count: {len(agent_with_tool.messages)}")
print()

for i, msg in enumerate(agent_with_tool.messages):
    role = msg['role']
    content_types = []
    
    # Identify what types of content are in each message
    for block in msg['content']:
        if 'text' in block:
            content_types.append(f"text: {block['text'][:60]}")
        elif 'toolUse' in block:
            tu = block['toolUse']
            content_types.append(f"toolUse: {tu['name']}({tu['input']})")
        elif 'toolResult' in block:
            tr = block['toolResult']
            # Extract the text from the tool result content
            result_text = ""
            for item in tr.get('content', []):
                if 'text' in item:
                    result_text = item['text'][:60]
            content_types.append(f"toolResult: {result_text}")
    
    print(f"[{i}] {role:10s} |")
    for ct in content_types:
        print(f"     {ct}")
    print()

### Understanding the 4 Messages

Here's what happened in each message and which part of the event loop created it:

| # | Role | Content | Created by | Event Loop Phase |
|---|------|---------|-----------|------------------|
| 0 | user | Your question | `_run_loop` (before event loop starts) | Setup |
| 1 | assistant | "I'll calculate..." + toolUse | `_handle_model_execution` in **Cycle 1** | Phase 1: Model |
| 2 | user | toolResult ("391") | `_handle_tool_execution` in **Cycle 1** | Phase 2: Tools |
| 3 | assistant | "17 * 23 = 391" | `_handle_model_execution` in **Cycle 2** | Phase 1: Model (after recurse) |

**The recursion happened between messages 2 and 3.** After the tool result was appended, `recurse_event_loop()` called `event_loop_cycle()` again. The model then saw the tool result and generated the final answer.

In [ ]:
# ============================================================
# Print event loop metrics
# ============================================================

# The metrics tell us about performance and how many cycles ran.
metrics = agent_with_tool.event_loop_metrics

print("=== Event Loop Metrics ===")
print(f"Input tokens:  {metrics.accumulated_usage.get('inputTokens', 'N/A')}")
print(f"Output tokens: {metrics.accumulated_usage.get('outputTokens', 'N/A')}")
print(f"Latency (ms):  {metrics.accumulated_metrics.get('latencyMs', 'N/A')}")

---
## Experiment 3: Multiple Tool Calls

What if the model needs to call multiple tools? Let's find out.

In [ ]:
# ============================================================
# EXPERIMENT 3: Two tools, model picks what it needs
# ============================================================

@tool
def multiply(a: int, b: int) -> str:
    """Multiply two numbers together.
    
    Args:
        a: The first number.
        b: The second number.
    """
    return str(a * b)

@tool
def add(a: int, b: int) -> str:
    """Add two numbers together.
    
    Args:
        a: The first number.
        b: The second number.
    """
    return str(a + b)

# Agent with both tools
agent_multi = Agent(
    tools=[multiply, add],
    callback_handler=None,
)

# Ask a question that requires both tools.
# The model might:
#   - Call multiply(17, 23) AND add(100, 200) in the same cycle
#   - OR call them in separate cycles
# Either way, the event loop handles it.

print("--- Calling agent (2 tools) ---")
result = agent_multi("What is 17 times 23, and also what is 100 plus 200?")
print(f"\n--- Result ---")
print(f"stop_reason: {result.stop_reason}")
print(f"response: {result}")

In [ ]:
# ============================================================
# Inspect the full message history for Experiment 3
# ============================================================

# If the model called both tools in one cycle:
#   [0] user:      your question
#   [1] assistant:  toolUse(multiply) + toolUse(add)
#   [2] user:       toolResult(391) + toolResult(300)
#   [3] assistant:  final answer
#
# If it called them in separate cycles:
#   [0] user:       your question
#   [1] assistant:  toolUse(multiply)
#   [2] user:       toolResult(391)
#   [3] assistant:  toolUse(add)
#   [4] user:       toolResult(300)
#   [5] assistant:  final answer

print(f"Total messages: {len(agent_multi.messages)}")
print()

for i, msg in enumerate(agent_multi.messages):
    role = msg['role']
    parts = []
    
    for block in msg['content']:
        if 'text' in block:
            parts.append(f"text: \"{block['text'][:50]}\"")
        elif 'toolUse' in block:
            tu = block['toolUse']
            parts.append(f"toolUse: {tu['name']}({tu['input']})")
        elif 'toolResult' in block:
            tr = block['toolResult']
            text = ""
            for item in tr.get('content', []):
                if 'text' in item:
                    text = item['text']
            parts.append(f"toolResult: \"{text}\"")
    
    print(f"[{i}] {role:10s}")
    for p in parts:
        print(f"     {p}")
    print()

---
## Stop Reasons Reference

| Stop Reason | What Happened | What Happens Next |
|-------------|--------------|-------------------|
| `"end_turn"` | Model finished speaking, no tool calls | Agent returns result to you |
| `"tool_use"` | Model wants to call one or more tools | Event loop executes tools, then recurses |
| `"max_tokens"` | Model hit token limit mid-response | `MaxTokensReachedException` thrown (error) |
| `"interrupt"` | Tool or hook triggered a pause | Agent returns with `interrupts` list |

**Most common flow:** model called -> `"tool_use"` -> tools run -> recurse -> model sees results -> `"end_turn"` -> done.

**Safety net:** The SDK has a `max_cycles` limit (default: 20) to prevent infinite loops where the model keeps calling tools forever.

---
## Summary

What you learned in this notebook:

- **The event loop** is the core engine: it calls the model, handles the response, executes tools if needed, and recurses
- **Without tools:** 1 cycle, 2 messages (user + assistant), stop_reason = `"end_turn"`
- **With tools:** 2+ cycles, 4+ messages (user + assistant/toolUse + toolResult + assistant), stop_reason = `"tool_use"` then `"end_turn"`
- **Recursion:** after tool execution, the event loop calls itself so the model can see tool results
- **Stop reasons:** `end_turn` (done), `tool_use` (need tools), `max_tokens` (error), `interrupt` (paused)
- **Key source:** `src/strands/event_loop/event_loop.py`

**Key source locations:**
| Function | Source |
|----------|--------|
| `event_loop_cycle` | `event_loop.py:78` |
| `_handle_model_execution` | `event_loop.py:275` |
| `_handle_tool_execution` | `event_loop.py:421` |
| `recurse_event_loop` | `event_loop.py:236` |

**Next:** [NB3_Models.ipynb](./NB3_Models.ipynb) -- Understand the model abstraction, streaming, and how to switch providers